In [ ]:
import time
from typing import List
import numpy as np
import torch
import torch.nn as nn
import torchode
from pathlib import Path
import pickle

from scipy.integrate import solve_ivp as sp_solve_ivp
from tqdm.auto import tqdm

from sklearn.preprocessing import MinMaxScaler
from ftnode.utils import set_global_seed, _load_loop_wrapper
from ftnode.node import (
    FTNODE, FeluSigmoidMLP,FeluSigmoidMLPfeaturized,
     GeluSigmoidMLPfeaturized,
)

import matplotlib.pyplot as plt
plt.style.use('default')
plt.rcParams['font.family']= 'serif'

device = 'cpu'
seed = 1234
set_global_seed(seed=seed)
random_state = 67

[Seed] Deterministic mode enabled (may reduce speed).


In [2]:
def hysteresis_ode(t,x,lam):
    return lam+x-x**3

In [3]:
n_lam = 51
n_traj = 51
lams = np.linspace(-1,1,n_lam)
xs = np.linspace(-2,2,n_traj)


In [4]:
t_max = 0.25
n_colloc = 101


Xs = []
Us = []
t = np.linspace(0,t_max,n_colloc)
for lami in tqdm(lams):
    for x0 in xs:
        sol = sp_solve_ivp(
            hysteresis_ode,
            t_span = [0,t_max],
            y0 = np.array(x0).reshape(-1),
            t_eval = np.linspace(0,t_max,n_colloc),
            args = (lami,)
        )

        Xs.append(sol.y.T)
        Us.append([lami])
Xs = np.array(Xs)
Us = np.array(Us)

  0%|          | 0/51 [00:00<?, ?it/s]

In [5]:
dXs = np.zeros_like(Xs)
T = t[np.newaxis,:,np.newaxis]
X_diff = Xs[:,2:,:] - Xs[:,:-2,:]
T_diff = T[:,2:,:] - T[:,:-2,:]

dXs[:,1:-1,:] = X_diff/T_diff
dXs[:,0,:] = (Xs[:,1,:] - Xs[:,0,:]) / (T[:,1,:] - T[:,0,:])
dXs[:,-1,:] = (Xs[:,-1,:] - Xs[:,-2,:]) / (T[:,-1,:] - T[:,-2,:])

In [6]:
scaler = MinMaxScaler(feature_range=(-1,1))
Xs_scaled = scaler.fit_transform(Xs.reshape(-1,1).reshape(-1,1)).reshape(-1,n_colloc,1)

In [7]:
dX_tensor = [
    torch.tensor(dxi,dtype=torch.float32,device=device) for dxi in dXs
]
X_tensor = [
    torch.tensor(xi,dtype=torch.float32,device=device) for xi in Xs_scaled
]
U_tensor = [
    torch.tensor(ui,dtype=torch.float32,device=device) for ui in Us
]

T_tensor = [
    torch.tensor(t,dtype=torch.float32, device=device) for _ in range(len(Xs))
]

In [8]:
class GradDataset(torch.utils.data.Dataset):
    def __init__(self, dX: List, X: List, T: List, U: List):
        self.dX = dX
        self.X = X
        self.T = T
        self.U = U
        # self.trans_idx = Transient_idx

    def __len__(self):
        return len(self.dX)

    def __getitem__(self, idx):
        if idx >= len(self):
            raise IndexError(
                f"Index {idx} is out of bounds of dataset size: {len(self)}."
            )

        dXi = self.dX[idx]
        Xi = self.X[idx]
        ti = self.T[idx]
        ui = self.U[idx]

        return dXi, Xi, ti, ui

dataset = GradDataset(dX = dX_tensor,X = X_tensor, T = T_tensor, U = U_tensor)

# Run $k$-Folds Cross-validation

In [9]:
k_folds = 10

In [10]:
from sklearn.model_selection import KFold
from torch.utils.data import DataLoader, SubsetRandomSampler
import time
import copy

In [11]:
avg_best_val_losses = []
avg_best_train_losses = []

In [12]:
# --- Configuration ---
k_folds = k_folds
n_epochs = 200  
batch_size = 50 
learning_rate = 1e-2
print_every = 10
_precision = 6
random_state = random_state
solve_method = 'tsit5'


kfold = KFold(n_splits=k_folds, shuffle=True, random_state=random_state)

val_results = []
train_results = []

def get_fresh_model_components():
    f = FeluSigmoidMLP(
        dims=[1,10,10,10, 1],
        activation=nn.SiLU(),
        lower_bound=-4,
        upper_bound=-0.1,
        init_type=None
    )


    g = GeluSigmoidMLPfeaturized(
        dims=[6, 10,10,10,1],
        activation=nn.SiLU(),
        lower_bound=-2,
        upper_bound=2,
        freq_sample_step=1,
        feat_lower_bound=-1.5,
        feat_upper_bound=1.5,
        init_type=None
    )

    model = FTNODE(f, g).to(device)
    return f, g, model 

# ==========================================
# K-Fold Loop
# ==========================================
for fold, (train_ids, val_ids) in enumerate(kfold.split(dataset)):
    print(f'\n--- FOLD {fold+1}/{k_folds} ---')

    fold_seed = random_state + fold
    set_global_seed(fold_seed)

    # 1. Re-initialize Model & Optimizer for this fold
    f_fold, g_fold, model_fold = get_fresh_model_components()
    model_fold.train() 
    
    loss_criteria = nn.MSELoss()
    

    opt = torch.optim.Adam(
        list(f_fold.parameters()) + list(g_fold.parameters()), 
        lr=learning_rate
    )
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=0.5, patience=10
    )

    # 2. Create DataLoaders for this fold
    train_subsampler = SubsetRandomSampler(train_ids)
    val_subsampler = SubsetRandomSampler(val_ids)

    trainloader = DataLoader(dataset, batch_size=batch_size, sampler=train_subsampler)
    valloader = DataLoader(dataset, batch_size=batch_size, sampler=val_subsampler)

    # 3. Training & Validation Loop
    best_val_loss = float('inf')
    fold_losses = []

    for epoch in tqdm(range(n_epochs), desc=f"Fold {fold+1}"):
        t1 = time.time()
        
        # --- TRAINING ---
        model_fold.train()
        train_loss = 0.0
        
        for batch_idx, (dXi, Xi, ti, ui) in enumerate(trainloader):
            x0i = Xi[:, 0, :]
            
            # ui_expanded = ui.unsqueeze(dim=1).expand(Xi.shape)
            # u_func = lambda t: ui_expanded
            u_func = lambda t: ui
            func = lambda t, x: model_fold(t, x, u_func)

            opt.zero_grad()
            sol = torchode.solve_ivp(
                f=func,
                y0=x0i,
                t_eval=ti.squeeze(),
                method=solve_method,
            )
            
            Xi_pred = sol.ys


            # loss = loss_criteria(dXi, dXi_pred)
            loss = loss_criteria(Xi, Xi_pred)
                
            loss.backward()
            opt.step()
            
            train_loss += loss.item()
        
        train_loss /= len(trainloader)

        # --- VALIDATION ---
        model_fold.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_idx, (dXi, Xi, ti, ui) in enumerate(valloader):
                x0i = Xi[:, 0, :]
                # ui_expanded = ui.unsqueeze(dim=1).expand(Xi.shape)
                # u_func = lambda t: ui_expanded
                u_func = lambda t: ui
                func = lambda t, x: model_fold(t, x, u_func)

                sol = torchode.solve_ivp(
                    f=func,
                    y0=x0i,
                    t_eval=ti.squeeze(),
                    method=solve_method,
                )

                Xi_pred = sol.ys


                # loss = loss_criteria(dXi, dXi_pred)
                loss = loss_criteria(Xi, Xi_pred)
                val_loss += loss.item()
        
        val_loss /= len(valloader)
        
        # --- SCHEDULER & LOGGING ---
        epoch_time = time.time() - t1
        
        # Step scheduler based on VALIDATION loss (standard practice)
        scheduler.step(val_loss) 
        cur_lr = opt.param_groups[0]['lr']

        if epoch <= 5 or epoch % print_every == 0 or epoch == n_epochs - 1:
            print(
                f"Epoch {epoch}: "
                f"Train Loss = {train_loss:.{_precision}e}, "
                f"Val Loss = {val_loss:.{_precision}e}, "
                f"Time = {epoch_time:.{_precision}e}, "
                f"lr = {cur_lr:.{_precision}e}"
            )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_train_loss = train_loss
            best_fold = fold
            
    print(f"Fold {fold+1} Best Val Loss: {best_val_loss:.{_precision}e}")
    val_results.append(best_val_loss)
    train_results.append(best_val_train_loss)

# --- SUMMARY ---
print("\nK-Fold Cross Validation Results:")
avg_loss = np.mean(val_results)
avg_train_loss = np.mean(train_results)
print(f"Average Best Validation Loss: {avg_loss:.{_precision}e}")
avg_best_val_losses.append(avg_loss)
avg_best_train_losses.append(avg_train_loss)
avg_best_val_losses, np.argmin(avg_best_val_losses)


--- FOLD 1/10 ---
[Seed] Deterministic mode enabled (may reduce speed).


Fold 1:   0%|          | 0/200 [00:00<?, ?it/s]

Epoch 0: Train Loss = 4.390168e-03, Val Loss = 1.342221e-03, Time = 3.337367e+00, lr = 1.000000e-02
Epoch 1: Train Loss = 8.726955e-04, Val Loss = 6.842670e-04, Time = 4.372928e+00, lr = 1.000000e-02
Epoch 2: Train Loss = 5.150348e-04, Val Loss = 5.245952e-04, Time = 3.978322e+00, lr = 1.000000e-02
Epoch 3: Train Loss = 3.749860e-04, Val Loss = 3.284333e-04, Time = 3.978350e+00, lr = 1.000000e-02
Epoch 4: Train Loss = 2.128250e-04, Val Loss = 1.410336e-04, Time = 3.628955e+00, lr = 1.000000e-02
Epoch 5: Train Loss = 9.914197e-05, Val Loss = 6.644733e-05, Time = 3.646021e+00, lr = 1.000000e-02
Epoch 10: Train Loss = 1.732593e-05, Val Loss = 1.177833e-05, Time = 3.406062e+00, lr = 1.000000e-02
Epoch 20: Train Loss = 5.224389e-06, Val Loss = 4.847794e-06, Time = 3.149774e+00, lr = 1.000000e-02
Epoch 30: Train Loss = 4.036168e-06, Val Loss = 3.445017e-06, Time = 3.227294e+00, lr = 1.000000e-02
Epoch 40: Train Loss = 2.274003e-06, Val Loss = 1.780549e-06, Time = 3.130705e+00, lr = 1.000000e

Fold 2:   0%|          | 0/200 [00:00<?, ?it/s]

Epoch 0: Train Loss = 4.366111e-03, Val Loss = 1.604272e-03, Time = 2.974709e+00, lr = 1.000000e-02
Epoch 1: Train Loss = 7.307559e-04, Val Loss = 3.458298e-04, Time = 3.497042e+00, lr = 1.000000e-02
Epoch 2: Train Loss = 2.338949e-04, Val Loss = 1.794578e-04, Time = 3.706236e+00, lr = 1.000000e-02
Epoch 3: Train Loss = 1.295999e-04, Val Loss = 1.060900e-04, Time = 3.790896e+00, lr = 1.000000e-02
Epoch 4: Train Loss = 7.962952e-05, Val Loss = 5.859884e-05, Time = 4.123400e+00, lr = 1.000000e-02
Epoch 5: Train Loss = 4.984689e-05, Val Loss = 4.646779e-05, Time = 3.593514e+00, lr = 1.000000e-02
Epoch 10: Train Loss = 1.250105e-05, Val Loss = 1.278948e-05, Time = 3.413614e+00, lr = 1.000000e-02
Epoch 20: Train Loss = 4.988841e-06, Val Loss = 6.353780e-06, Time = 3.377863e+00, lr = 1.000000e-02
Epoch 30: Train Loss = 5.863545e-06, Val Loss = 5.350791e-06, Time = 3.246891e+00, lr = 1.000000e-02
Epoch 40: Train Loss = 2.339057e-06, Val Loss = 3.755180e-06, Time = 3.331469e+00, lr = 1.000000e

...

Fold 9:   0%|          | 0/200 [00:00<?, ?it/s]

Epoch 0: Train Loss = 4.358292e-03, Val Loss = 1.871498e-03, Time = 3.097083e+00, lr = 1.000000e-02
Epoch 1: Train Loss = 1.030721e-03, Val Loss = 5.587076e-04, Time = 3.628501e+00, lr = 1.000000e-02
Epoch 2: Train Loss = 3.701540e-04, Val Loss = 2.255528e-04, Time = 3.709287e+00, lr = 1.000000e-02
Epoch 3: Train Loss = 1.432453e-04, Val Loss = 9.394454e-05, Time = 3.566648e+00, lr = 1.000000e-02
Epoch 4: Train Loss = 7.309645e-05, Val Loss = 5.083157e-05, Time = 3.411191e+00, lr = 1.000000e-02
Epoch 5: Train Loss = 4.868317e-05, Val Loss = 4.067945e-05, Time = 3.571924e+00, lr = 1.000000e-02
Epoch 10: Train Loss = 1.388476e-05, Val Loss = 1.096343e-05, Time = 3.435335e+00, lr = 1.000000e-02
Epoch 20: Train Loss = 3.394492e-06, Val Loss = 2.408617e-06, Time = 3.254136e+00, lr = 1.000000e-02
Epoch 30: Train Loss = 1.136651e-06, Val Loss = 1.377568e-06, Time = 3.194564e+00, lr = 1.000000e-02
Epoch 40: Train Loss = 2.417332e-06, Val Loss = 6.001401e-06, Time = 3.308073e+00, lr = 1.000000e

Fold 10:   0%|          | 0/200 [00:00<?, ?it/s]

Epoch 0: Train Loss = 3.976058e-03, Val Loss = 1.493986e-03, Time = 3.340674e+00, lr = 1.000000e-02
Epoch 1: Train Loss = 9.070685e-04, Val Loss = 4.012732e-04, Time = 3.661471e+00, lr = 1.000000e-02
Epoch 2: Train Loss = 3.156455e-04, Val Loss = 1.840019e-04, Time = 3.755366e+00, lr = 1.000000e-02
Epoch 3: Train Loss = 1.592931e-04, Val Loss = 1.276423e-04, Time = 3.964097e+00, lr = 1.000000e-02
Epoch 4: Train Loss = 9.341000e-05, Val Loss = 7.040214e-05, Time = 3.906793e+00, lr = 1.000000e-02
Epoch 5: Train Loss = 5.841118e-05, Val Loss = 4.985566e-05, Time = 3.793749e+00, lr = 1.000000e-02
Epoch 10: Train Loss = 2.186458e-05, Val Loss = 1.611730e-05, Time = 3.705582e+00, lr = 1.000000e-02
Epoch 20: Train Loss = 8.354143e-06, Val Loss = 7.326009e-06, Time = 3.364478e+00, lr = 1.000000e-02
Epoch 30: Train Loss = 3.923119e-06, Val Loss = 5.087157e-06, Time = 3.338002e+00, lr = 1.000000e-02
Epoch 40: Train Loss = 4.017592e-06, Val Loss = 2.522168e-06, Time = 3.232850e+00, lr = 1.000000e

([9.600668414340891e-08], 0)